<a href="https://colab.research.google.com/github/matstabel/deep_learning/blob/main/ML_Exercise5_tune_solution.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Assignment: Hyperparameter Tuning with Keras Tuner

In this assignment, you'll practice automated hyperparameter tuning using Keras Tuner by building and optimizing a simple neural network model.

### Objective
Create a neural network model using Keras Tuner to systematically search for optimal hyperparameters. Specifically, you will search over:

- **Number of neurons:** Search among reasonable numbers of neurons in the hidden layer.
- **Learning rate:** Search among learning rates of `1e-2`, `1e-3`, or `1e-4`.
- **Architecture:** Use exactly **one hidden layer**.
- **Optimizer:** Use the Adam optimizer.

### Instructions

1. **Data Preparation**: Use Fashion MNIST.

2. **Define the Model**: Use the following template for your `build_model` function:

```python
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
    units = hp.Choice(name="units", values=["YOUR CODE HERE"])
    learning_rate = hp.Choice(name="learning_rate", values=[1e-2, 1e-3, 1e-4])

    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"])

    return model
```

3. **Hyperparameter Search**: Use Keras Tuner (`BayesianOptimization`) to run the search for optimal hyperparameters. Set `keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)` and examine what this does. Record the top 4 configurations.

4. **Optimal number of epochs**: Find the optimal number of epochs using the best training configuration.

5. **Train on the full dataset and test**

In [ ]:
!pip install keras-tuner -q

**Explanation for the number of neurons**: Using the bottle-neck principle, we can reason as follows: We have 784 inputs, so that's our maximum and 10 output neurons so that's our minimum. By examining the input images, we can tell that that a fair amount of neurons are non-zero. This indicates that a number of neurons in the large range is probably best. Also think about regular Mnist where 64 neurons is probably the lowest we should go. This is a more complicated example meaning a more complicated model is probably warranted. We then go from 128 to 256 to 512 following the powers-of-2-rule to ensure that our model fits nicely into computer memory.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

def build_model(hp):
    units = hp.Choice("units", [128, 256, 512])
    learning_rate = hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])

    model = keras.Sequential([
        layers.Dense(units, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    model.compile(
        optimizer=optimizer,
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [ ]:
import kerastuner as kt

tuner = kt.BayesianOptimization(
    build_model,
    objective="val_accuracy",
    max_trials=5,
    executions_per_trial=2,
    directory="mnist_kt_test",
    overwrite=True,
)

In [ ]:
# 1. Load the Fashion MNIST dataset
import tensorflow as tf
fashion_mnist = tf.keras.datasets.fashion_mnist
(train_images, train_labels), (test_images, test_labels) = fashion_mnist.load_data()

# 2. Preprocess the data:
# Normalize the images to a 0-1 range
train_images = train_images.reshape((60000, 28 * 28))
train_images = train_images.astype("float32") / 255
test_images = test_images.reshape((10000, 28 * 28))
test_images = test_images.astype("float32") / 255

`keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)` creates a Keras callback called `ReduceLROnPlateau`, which is used to automatically reduce the learning rate during training when the model stops improving.

It watches the validation loss (`val_loss`) during training. If the validation loss stops improving, the callback will act.

When triggered, the callback will reduce the learning rate by this factor (`factor=0.5`). So if the current learning rate is `0.001`, it will be changed to `0.001 * 0.5 = 0.0005`.

`patience=3` - if `val_loss` hasn't improved for 3 consecutive epochs, the learning rate will be reduced.

Sometimes during training, the model hits a plateau — it stops improving. Instead of getting stuck, lowering the learning rate can help the model fine-tune its weights and keep learning.

In [ ]:
# Perform search
tuner.search(
    train_images, train_labels,
    batch_size=128,
    epochs=100,
    validation_split=0.2,
    callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=5),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
    ],
    verbose=2,
)

Trial 5 Complete [00h 14m 02s]
val_accuracy: 0.893666684627533

Best val_accuracy So Far: 0.9033749997615814
Total elapsed time: 00h 41m 55s


In [ ]:
top_n = 4
best_hps = tuner.get_best_hyperparameters(top_n)

In [ ]:
def get_best_epoch(hp):
    model = build_model(hp)
    history = model.fit(
      train_images, train_labels,
      batch_size=128,
      epochs=100,
      validation_split=0.2,
      callbacks=[
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=10),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3)
    ])
    val_loss_per_epoch = history.history["val_loss"]
    best_epoch = val_loss_per_epoch.index(min(val_loss_per_epoch)) + 1
    print(f"Best epoch: {best_epoch}")
    return best_epoch

In [ ]:
def get_best_trained_model(hp):
    best_epoch = get_best_epoch(hp)
    model = build_model(hp)
    model.fit(
        train_images, train_labels,
        batch_size=128, epochs=int(best_epoch * 1.2))
    return model

best_models = []
for hp in best_hps:
    model = get_best_trained_model(hp)
    model.evaluate(test_images, test_labels)
    best_models.append(model)

Epoch 1/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step - accuracy: 0.7592 - loss: 0.6900 - val_accuracy: 0.8553 - val_loss: 0.4178 - learning_rate: 0.0010
Epoch 2/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.8618 - loss: 0.3964 - val_accuracy: 0.8675 - val_loss: 0.3736 - learning_rate: 0.0010
Epoch 3/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.8746 - loss: 0.3505 - val_accuracy: 0.8697 - val_loss: 0.3682 - learning_rate: 0.0010
Epoch 4/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8857 - loss: 0.3186 - val_accuracy: 0.8785 - val_loss: 0.3377 - learning_rate: 0.0010
Epoch 5/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.8911 - loss: 0.2956 - val_accuracy: 0.8737 - val_loss: 0.3468 - learning_rate: 0.0010
Epoch 6/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 15ms/step - accuracy: 0.8979 - loss: 0.2796 - val_accuracy: 0.8850 - val_loss: 0.3165 - learning_rate: 0.0010
Epoch 7/100
375/375 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9025 - l

In [ ]:
for idx, model in enumerate(best_models):
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    print(f"Model {idx+1}: Test Loss = {loss:.4f}, Test Accuracy = {acc:.4f}")


Model 1: Test Loss = 0.3424, Test Accuracy = 0.8934
Model 2: Test Loss = 0.3430, Test Accuracy = 0.8914
Model 3: Test Loss = 0.3099, Test Accuracy = 0.8957
Model 4: Test Loss = 0.3168, Test Accuracy = 0.8888


In [ ]:
import pandas as pd

results = []
for idx, (hp, model) in enumerate(zip(best_hps, best_models)):
    loss, acc = model.evaluate(test_images, test_labels, verbose=0)
    results.append({
        "Model": idx+1,
        "Units": hp.get("units"),
        "Learning Rate": hp.get("learning_rate"),
        "Test Accuracy": f"{acc*100:.2f}%",
        "Test Loss": f"{loss:.3f}"
    })

df_results = pd.DataFrame(results)
print(df_results)

   Model  Units  Learning Rate Test Accuracy Test Loss
0      1    512         0.0010        89.34%     0.342
1      2    256         0.0010        89.14%     0.343
2      3    512         0.0001        89.57%     0.310
3      4    256         0.0001        88.88%     0.317
